# LeetCode 2232 — Minimize Result by Adding Parentheses to Expression bằng Python

Link đề: https://leetcode.com/problems/minimize-result-by-adding-parentheses-to-expression/ · Độ khó: **Medium** · Chủ đề: String, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def minimizeResult(self, expression: str) -> str:
        ...
```

## 1. Mục tiêu học

1. Hiểu **ngữ nghĩa** của biểu thức sau khi thêm ngoặc: `A(B+C)D` nghĩa là `A × (B + C) × D`, trong đó phần **rỗng** bên ngoài ngoặc đóng vai trò **1** (phần tử trung hòa của phép nhân).
2. **Liệt kê** mọi vị trí đặt ngoặc trái (trong số thứ nhất) và ngoặc phải (trong số thứ hai) — chỉ `O(len²)` cách vì độ dài ≤ 10.
3. Tính giá trị **bằng số học** (`int` của các lát cắt chuỗi) thay vì gọi `eval` — nhanh và an toàn.
4. Tự viết một **bộ tính biểu thức** nhỏ (có phép nhân ngầm định) để kiểm chứng độc lập.
5. Kiểm thử bài **nhiều đáp án** bằng hàm kiểm tra: chuỗi trả về hợp lệ về định dạng **và** có giá trị nhỏ nhất.

## 2. Đề bài & ví dụ

Cho chuỗi `expression` dạng `"<num1>+<num2>"` (hai số nguyên dương). Thêm **một cặp ngoặc** sao cho biểu thức hợp lệ và có **giá trị nhỏ nhất**: ngoặc trái đặt **bên trái** dấu `+` (trong `num1`, có thể ngay đầu), ngoặc phải đặt **bên phải** dấu `+` (trong `num2`, có thể ngay cuối). Phần số ngay ngoài ngoặc được **nhân** với biểu thức trong ngoặc.

Trả về biểu thức sau khi thêm ngoặc (nếu nhiều cách cùng giá trị nhỏ nhất, trả về bất kỳ).

**Ví dụ 1:** `"247+38"` → `"2(47+38)"` = `2 × 85 = 170`. (`"2(4)7+38"` không hợp lệ vì ngoặc phải nằm bên trái `+`.)

**Ví dụ 2:** `"12+34"` → `"1(2+3)4"` = `1 × 5 × 4 = 20`.

**Ví dụ 3:** `"999+999"` → `"(999+999)"` = `1998`.

**Ràng buộc:** `3 ≤ len(expression) ≤ 10`, chỉ chữ số `1..9` và đúng một dấu `+`; mọi giá trị vừa `int32`.

In [1]:
import random
import time
from itertools import product
from typing import List
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)   # eval('2(...)') sinh SyntaxWarning ra stderr

examples = [("247+38", "2(47+38)", 170), ("12+34", "1(2+3)4", 20), ("999+999", "(999+999)", 1998)]
for e, out, val in examples:
    print(f"{e!r} -> {out!r} (giá trị {val})")

'247+38' -> '2(47+38)' (giá trị 170)
'12+34' -> '1(2+3)4' (giá trị 20)
'999+999' -> '(999+999)' (giá trị 1998)


## 3. Kiến thức nền

### 3.1 Tách chuỗi theo dấu `+`

`expression.index('+')` cho vị trí dấu cộng `p`; `num1 = expression[:p]`, `num2 = expression[p+1:]`. Hoặc `a, b = expression.split('+')`.

In [2]:
e = "247+38"
p = e.index("+")
print("p =", p, "| num1 =", e[:p], "| num2 =", e[p + 1:], "| split:", e.split("+"))

p = 3 | num1 = 247 | num2 = 38 | split: ['247', '38']


### 3.2 Cắt số thành "phần ngoài" và "phần trong"

Đặt ngoặc trái trước vị trí `i` của `num1` (`0 ≤ i < len(num1)`): phần ngoài bên trái `num1[:i]`, phần trong `num1[i:]`. Đặt ngoặc phải sau vị trí `j` của `num2` (`0 ≤ j < len(num2)`): phần trong `num2[:j+1]`, phần ngoài bên phải `num2[j+1:]`. Phần trong luôn **khác rỗng** (ngoặc phải bao ít nhất một chữ số mỗi bên).

In [3]:
num1, num2 = "247", "38"
for i in range(len(num1)):
    print(f"i={i}: ngoài-trái={num1[:i]!r}, trong-trái={num1[i:]!r}")
for j in range(len(num2)):
    print(f"j={j}: trong-phải={num2[:j + 1]!r}, ngoài-phải={num2[j + 1:]!r}")

i=0: ngoài-trái='', trong-trái='247'
i=1: ngoài-trái='2', trong-trái='47'
i=2: ngoài-trái='24', trong-trái='7'
j=0: trong-phải='3', ngoài-phải='8'
j=1: trong-phải='38', ngoài-phải=''


### 3.3 Phần rỗng = 1 (phần tử trung hòa)

`"(47+38)"` không có số bên trái ⇒ nhân với **1**, không phải 0. Viết `int(s) if s else 1`.

In [4]:
def as_factor(s):
    return int(s) if s else 1

print(as_factor(""), as_factor("2"), as_factor("12"))

1 2 12


### 3.4 Vì sao không dùng thẳng `eval`?

`eval("2(47+38)")` lỗi: Python hiểu `2(...)` là "gọi hàm" trên số 2. Phải chèn `*` trước khi `eval`, và `eval` trên dữ liệu ngoài là thói quen **nguy hiểm**. Ở đây ta chỉ dùng `eval` trong brute force để kiểm tra (dữ liệu do ta tự sinh), còn lời giải chính tính bằng số học.

In [5]:
try:
    eval("2(47+38)")
except TypeError as err:
    print("eval trực tiếp lỗi:", err)
print("chèn '*' rồi eval:", eval("2*(47+38)"))

eval trực tiếp lỗi: 'int' object is not callable
chèn '*' rồi eval: 170


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `"12+34"`: `num1 = "12"`, `num2 = "34"`. Mỗi số có 2 vị trí đặt ngoặc ⇒ `2 × 2 = 4` cách:

| Biểu thức | Giá trị |
|---|---|
| `(12+3)4` | `15 × 4 = 60` |
| `(12+34)` | `46` |
| `1(2+3)4` | `1 × 5 × 4 = 20` ← nhỏ nhất |
| `1(2+34)` | `1 × 36 = 36` |

Trực giác: đặt ngoặc "bên trong" làm tổng nhỏ đi nhưng tạo hệ số nhân; không có quy luật đơn giản ⇒ cứ **thử hết** (tối đa `9 × 9 = 81` cách khi độ dài 10).

In [6]:
num1, num2 = "12", "34"
for i in range(len(num1)):
    for j in range(len(num2)):
        L, A, B, R = num1[:i], num1[i:], num2[:j + 1], num2[j + 1:]
        val = as_factor(L) * (int(A) + int(B)) * as_factor(R)
        print(f"{L}({A}+{B}){R}: {as_factor(L)} × ({A}+{B}) × {as_factor(R)} = {val}")

(12+3)4: 1 × (12+3) × 4 = 60
(12+34): 1 × (12+34) × 1 = 46
1(2+3)4: 1 × (2+3) × 4 = 20
1(2+34): 1 × (2+34) × 1 = 36


### 4.1 Hai lực kéo ngược chiều

Khi dịch ngoặc trái sang phải (tăng `i`):
- **Tổng trong ngoặc giảm** (vì `num1[i:]` ngắn đi, mất chữ số hàng cao).
- **Thừa số bên trái tăng** (vì `num1[:i]` dài ra) — từ 1 lên một số có `i` chữ số.

Tương tự với ngoặc phải. Vì thế không thể nói "càng đẩy ngoặc vào trong càng tốt" hay ngược lại. Ví dụ `"999+999"`: mọi cách đặt ngoặc vào trong đều tạo thừa số ≥ 9 mà tổng chỉ giảm chưa tới 10 lần ⇒ bao trọn là tốt nhất. Ngược lại `"12+34"`: tách `1(2+3)4` giảm tổng từ 46 xuống 5 (gấp 9 lần) mà thừa số chỉ là `1 × 4` ⇒ có lợi.

### 4.2 Quan sát hữu ích: chữ số `1` ở đầu là "miễn phí"

Nếu `num1` bắt đầu bằng `1` và đặt ngoặc ngay sau nó thì thừa số trái là `1` — không tốn gì mà vẫn bỏ được chữ số hàng cao khỏi tổng. Đây là lý do ví dụ 2 có đáp án `1(2+3)4`. Đoạn code dưới đếm, trên các biểu thức ngẫu nhiên, tỷ lệ đáp án tối ưu là **bao trọn** `(num1+num2)`.

In [7]:
def best_value(expression):
    num1, num2 = expression.split("+")
    return min((int(num1[:i]) if i else 1) * (int(num1[i:]) + int(num2[:j + 1])) * (int(num2[j + 1:]) if j + 1 < len(num2) else 1)
               for i in range(len(num1)) for j in range(len(num2)))

rng0 = random.Random(1)
full = 0; total = 600
for _ in range(total):
    a = rng0.randint(1, 4); b = rng0.randint(1, 4)
    e = "".join(rng0.choice("123456789") for _ in range(a)) + "+" + "".join(rng0.choice("123456789") for _ in range(b))
    num1, num2 = e.split("+")
    if best_value(e) == int(num1) + int(num2):
        full += 1
print(f"Bao trọn là tối ưu ở {full}/{total} biểu thức ngẫu nhiên -> vẫn phải xét mọi vị trí")

Bao trọn là tối ưu ở 186/600 biểu thức ngẫu nhiên -> vẫn phải xét mọi vị trí


## 5. Cách 1 — Brute Force / baseline: sinh chuỗi rồi `eval` (có chèn `*`)

**Ý tưởng:** với mọi vị trí ngoặc, dựng chuỗi kết quả, dựng chuỗi Python tương đương (chèn `*` ở chỗ nhân ngầm), `eval`, lấy giá trị nhỏ nhất.

In [8]:
def to_python(expr_with_paren):
    out = []
    for idx, ch in enumerate(expr_with_paren):
        if ch == "(" and idx > 0: out.append("*")
        out.append(ch)
        if ch == ")" and idx < len(expr_with_paren) - 1: out.append("*")
    return "".join(out)

def minimize_eval(expression):
    p = expression.index("+")
    best_val, best_str = None, None
    for i in range(p):
        for j in range(p + 1, len(expression)):
            s = expression[:i] + "(" + expression[i:j + 1] + ")" + expression[j + 1:]
            v = eval(to_python(s))
            if best_val is None or v < best_val:
                best_val, best_str = v, s
    return best_str

for e, out, val in examples:
    got = minimize_eval(e); print(e, "->", got, "=", eval(to_python(got))); assert eval(to_python(got)) == val

247+38 -> 2(47+38) = 170
12+34 -> 1(2+3)4 = 20
999+999 -> (999+999) = 1998


**Độ phức tạp:** `O(L²)` vị trí × `O(L)` dựng chuỗi + `eval` (tốn kém: phân tích cú pháp Python) — với `L ≤ 10` vẫn rất nhanh, nhưng hằng số lớn và dùng `eval`.

**Nút thắt:** `eval` chậm và không an toàn; giá trị có thể tính trực tiếp bằng `int` của bốn lát cắt.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với vị trí ngoặc `(i, j)`:

$$\text{giá trị} = f(\text{num1}[:i]) \times \big(\text{int}(\text{num1}[i:]) + \text{int}(\text{num2}[:j+1])\big) \times f(\text{num2}[j+1:]),$$

với `f(s) = int(s)` nếu `s` khác rỗng, ngược lại `1`.

**Quy tắc "nếu… thì…":**
- **Nếu** `i = 0` **thì** không có thừa số bên trái (`× 1`).
- **Nếu** `j = len(num2) − 1` **thì** không có thừa số bên phải.
- **Nếu** giá trị nhỏ hơn kỷ lục **thì** lưu `(i, j)`.

**Không có cấu trúc đơn điệu** đơn giản (giá trị không đơn điệu theo `i` hay `j`), nhưng không gian chỉ `≤ 81` cặp ⇒ liệt kê là tối ưu thực tế.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Liệt kê + tính bằng số học (bản nộp chính)

In [9]:
def minimize_arith(expression):
    num1, num2 = expression.split("+")
    best = None
    for i in range(len(num1)):
        left = int(num1[:i]) if i else 1
        a = int(num1[i:])
        for j in range(len(num2)):
            b = int(num2[:j + 1])
            right = int(num2[j + 1:]) if j + 1 < len(num2) else 1
            v = left * (a + b) * right
            if best is None or v < best[0]:
                best = (v, i, j)
    _, i, j = best
    return f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"

for e, out, val in examples:
    assert minimize_arith(e) == out
print([minimize_arith(e) for e, _, _ in examples])

['2(47+38)', '1(2+3)4', '(999+999)']


**Độ phức tạp Cách 2:** `O(L²)` cặp × `O(L)` cho `int(...)` ⇒ `O(L³)` với `L ≤ 10` ⇒ vài trăm phép tính. Space `O(L)`.

### 7.2 Cách 3 — Tiền tính các thừa số (mỗi lát cắt `int` một lần)

Tính trước `left[i]`, `inner_left[i]` cho mọi `i` và `inner_right[j]`, `right[j]` cho mọi `j` ⇒ vòng lặp đôi chỉ còn phép nhân/cộng `O(1)`.

In [10]:
def minimize_precomputed(expression):
    num1, num2 = expression.split("+")
    left = [int(num1[:i]) if i else 1 for i in range(len(num1))]
    inl = [int(num1[i:]) for i in range(len(num1))]
    inr = [int(num2[:j + 1]) for j in range(len(num2))]
    right = [int(num2[j + 1:]) if j + 1 < len(num2) else 1 for j in range(len(num2))]
    v, i, j = min((left[i] * (inl[i] + inr[j]) * right[j], i, j) for i in range(len(num1)) for j in range(len(num2)))
    return f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"

for e, out, val in examples:
    assert minimize_precomputed(e) == out
print([minimize_precomputed(e) for e, _, _ in examples])

['2(47+38)', '1(2+3)4', '(999+999)']


**Độ phức tạp Cách 3:** tiền tính `O(L²)` (mỗi `int` lát cắt `O(L)`), vòng đôi `O(L²)`.

### 7.3 Cách 4 — Tự viết bộ tính biểu thức (không dùng `eval`)

Để kiểm chứng độc lập, viết hàm tính giá trị của chuỗi dạng `A(B+C)D`: đọc số trước `(`, trong ngoặc tách `+`, sau `)`. Hàm này kiểm cả **định dạng hợp lệ** — dùng trong bộ test.

In [11]:
def evaluate(s):
    """Trả về giá trị của s dạng L(A+B)R, hoặc None nếu sai định dạng."""
    if s.count("(") != 1 or s.count(")") != 1 or s.count("+") != 1:
        return None
    lp, rp, plus = s.index("("), s.index(")"), s.index("+")
    if not (lp < plus < rp):
        return None
    L, A, B, R = s[:lp], s[lp + 1:plus], s[plus + 1:rp], s[rp + 1:]
    if not A or not B or not all(x.isdigit() for x in (A, B)) or any(x and not x.isdigit() for x in (L, R)):
        return None
    return (int(L) if L else 1) * (int(A) + int(B)) * (int(R) if R else 1)

print([evaluate(o) for _, o, _ in examples], evaluate("2(4)7+38"), evaluate("(247+38)"))

[170, 20, 1998] None 285


In [12]:
class SolutionArith:
    def minimizeResult(self, expression: str) -> str:
        num1, num2 = expression.split("+")
        best_v, best = float("inf"), ""
        for i in range(len(num1)):
            for j in range(1, len(num2) + 1):
                l = int(num1[:i]) if i else 1
                r = int(num2[j:]) if j < len(num2) else 1
                v = l * (int(num1[i:]) + int(num2[:j])) * r
                if v < best_v:
                    best_v, best = v, num1[:i] + "(" + num1[i:] + "+" + num2[:j] + ")" + num2[j:]
        return best

print([SolutionArith().minimizeResult(e) for e, _, _ in examples])

['2(47+38)', '1(2+3)4', '(999+999)']


**Độ phức tạp Cách 4:** như Cách 2. (`j` ở đây là số chữ số của `num2` nằm trong ngoặc, từ 1 tới `len(num2)` — cách đánh chỉ số khác nhưng tương đương.)

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 `"247+38"` (`num1 = "247"`, `num2 = "38"`):

| i | j | trái | trong (a + b) | phải | giá trị | kỷ lục |
|---|---|---|---|---|---|---|
| 0 | 0 | 1 | 247 + 3 | 8 | 2000 | 2000 |
| 0 | 1 | 1 | 247 + 38 | 1 | 285 | 285 |
| 1 | 0 | 2 | 47 + 3 | 8 | 800 | 285 |
| 1 | 1 | 2 | 47 + 38 | 1 | **170** | 170 |
| 2 | 0 | 24 | 7 + 3 | 8 | 1920 | 170 |
| 2 | 1 | 24 | 7 + 38 | 1 | 1080 | 170 |

In [13]:
def trace_arith(expression):
    num1, num2 = expression.split("+"); best = None
    for i in range(len(num1)):
        for j in range(len(num2)):
            left = int(num1[:i]) if i else 1; a = int(num1[i:]); b = int(num2[:j + 1])
            right = int(num2[j + 1:]) if j + 1 < len(num2) else 1
            v = left * (a + b) * right
            new = best is None or v < best[0]
            if new: best = (v, i, j)
            print(f"i={i} j={j}: {num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]:<4} = {left} × ({a}+{b}) × {right} = {v:5d}"
                  + ("  <- kỷ lục mới" if new else ""))
    _, i, j = best
    return f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"

print("Đáp án:", trace_arith("247+38"))

i=0 j=0: (247+3)8    = 1 × (247+3) × 8 =  2000  <- kỷ lục mới
i=0 j=1: (247+38)     = 1 × (247+38) × 1 =   285  <- kỷ lục mới
i=1 j=0: 2(47+3)8    = 2 × (47+3) × 8 =   800
i=1 j=1: 2(47+38)     = 2 × (47+38) × 1 =   170  <- kỷ lục mới
i=2 j=0: 24(7+3)8    = 24 × (7+3) × 8 =  1920
i=2 j=1: 24(7+38)     = 24 × (7+38) × 1 =  1080
Đáp án: 2(47+38)


In [14]:
print("Đáp án:", trace_arith("999+999"))

i=0 j=0: (999+9)99   = 1 × (999+9) × 99 = 99792  <- kỷ lục mới
i=0 j=1: (999+99)9    = 1 × (999+99) × 9 =  9882  <- kỷ lục mới
i=0 j=2: (999+999)     = 1 × (999+999) × 1 =  1998  <- kỷ lục mới
i=1 j=0: 9(99+9)99   = 9 × (99+9) × 99 = 96228
i=1 j=1: 9(99+99)9    = 9 × (99+99) × 9 = 16038
i=1 j=2: 9(99+999)     = 9 × (99+999) × 1 =  9882
i=2 j=0: 99(9+9)99   = 99 × (9+9) × 99 = 176418
i=2 j=1: 99(9+99)9    = 99 × (9+99) × 9 = 96228
i=2 j=2: 99(9+999)     = 99 × (9+999) × 1 = 99792
Đáp án: (999+999)


### 8.1 Trace ví dụ 2 với bảng giá trị theo (i, j)

Ta in ma trận giá trị: hàng là `i` (vị trí ngoặc trái), cột là `j` (vị trí ngoặc phải). Ô nhỏ nhất là đáp án. Cách nhìn ma trận này giúp thấy rõ giá trị **không đơn điệu** theo hàng hay cột.

In [15]:
def value_matrix(expression):
    num1, num2 = expression.split("+")
    M = [[(int(num1[:i]) if i else 1) * (int(num1[i:]) + int(num2[:j + 1])) * (int(num2[j + 1:]) if j + 1 < len(num2) else 1)
          for j in range(len(num2))] for i in range(len(num1))]
    print(f"{expression}: hàng i = vị trí '(' trong {num1!r}, cột j = vị trí ')' trong {num2!r}")
    for i, row in enumerate(M):
        print(f"  i={i} ({num1[:i] or '-':>4}|{num1[i:]:<4}):", " ".join(f"{x:6d}" for x in row))
    return M

M = value_matrix("12+34")
print("min =", min(min(r) for r in M))
M = value_matrix("1234+5678")
print("min =", min(min(r) for r in M), "->", Solution().minimizeResult("1234+5678") if "Solution" in globals() else minimize_arith("1234+5678"))

12+34: hàng i = vị trí '(' trong '12', cột j = vị trí ')' trong '34'
  i=0 (   -|12  ):     60     46
  i=1 (   1|2   ):     20     36
min = 20
1234+5678: hàng i = vị trí '(' trong '1234', cột j = vị trí ')' trong '5678'
  i=0 (   -|1234): 840042 100620  14408   6912
  i=1 (   1|234 ): 162042  22620   6408   5912
  i=2 (  12|34  ): 317304  84240  57696  68544
  i=3 ( 123|4   ): 750546 575640 561864 698886
min = 5912 -> 1(234+5678)


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minimizeResult(self, expression: str) -> str:
        num1, num2 = expression.split("+")
        best_val, best_expr = float("inf"), ""
        for i in range(len(num1)):                       # ngoặc trái trước num1[i]
            left = int(num1[:i]) if i else 1             # phần ngoài rỗng -> nhân 1
            for j in range(len(num2)):                   # ngoặc phải sau num2[j]
                right = int(num2[j + 1:]) if j + 1 < len(num2) else 1
                val = left * (int(num1[i:]) + int(num2[:j + 1])) * right
                if val < best_val:
                    best_val = val
                    best_expr = f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"
        return best_expr

for e, out, val in examples:
    assert Solution().minimizeResult(e) == out
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Không gian lời giải.** Một cách đặt ngoặc hợp lệ được xác định duy nhất bởi `(i, j)`: ngoặc trái ngay trước chữ số `num1[i]` (`0 ≤ i < |num1|`, vì ngoặc trái phải ở bên trái `+` và bao ít nhất một chữ số của `num1`), ngoặc phải ngay sau chữ số `num2[j]` (`0 ≤ j < |num2|`). Ngược lại mọi `(i, j)` như vậy cho một biểu thức hợp lệ. Vòng lặp đôi duyệt **đúng** tập này.

**Giá trị.** Theo ngữ nghĩa đề (phép nhân ngầm định giữa số và ngoặc), biểu thức `L(A+B)R` có giá trị `L·(A+B)·R`, với `L`/`R` rỗng là 1. Công thức trong code tính đúng điều đó.

**Kết luận.** Lấy min trên toàn bộ không gian ⇒ đúng giá trị nhỏ nhất; chuỗi trả về tương ứng với cặp `(i, j)` đạt min. Bộ test so giá trị với hàm `evaluate` độc lập và với bản dùng `eval`.

## 11. Phân tích Time / Space

Gọi `L = len(expression) ≤ 10`, `a = |num1|`, `b = |num2|` (`a + b = L − 1`).

| Cách | Time | Space |
|---|---|---|
| 1. Sinh chuỗi + `eval` | `O(a·b)` lần `eval` (mỗi lần `O(L)` nhưng hằng số lớn) | `O(L)` |
| 2. Số học trực tiếp | `O(a·b·L)` | `O(L)` |
| 3. Tiền tính thừa số | `O(L² + a·b)` | `O(L)` |
| 4. Bộ tính tự viết | `O(a·b·L)` | `O(L)` |

Với `L ≤ 10`, `a·b ≤ 25` (vì `a + b ≤ 9` nên max `a·b` là `4·5 = 20`). Mọi cách đều tức thì; đo để so hằng số:

In [17]:
rng = random.Random(2232)
cases = []
for _ in range(2000):
    a = rng.randint(1, 8); b = rng.randint(1, 9 - a)
    cases.append("".join(rng.choice("123456789") for _ in range(a)) + "+" + "".join(rng.choice("123456789") for _ in range(b)))
for name, fn in [("eval", minimize_eval), ("arith", minimize_arith), ("precomputed", minimize_precomputed),
                 ("Solution", lambda e: Solution().minimizeResult(e))]:
    t = time.perf_counter()
    for e in cases: fn(e)
    print(f"{name:11s}: 2000 biểu thức trong {time.perf_counter() - t:.3f}s")

eval       : 2000 biểu thức trong 0.130s


arith      : 2000 biểu thức trong 0.006s
precomputed: 2000 biểu thức trong 0.006s
Solution   : 2000 biểu thức trong 0.007s


## 12. Bộ test

Vì có thể có nhiều đáp án cùng giá trị, kiểm: (a) chuỗi trả về **hợp lệ** (đúng định dạng, bỏ ngoặc thì ra đúng `expression`), (b) giá trị **bằng** giá trị nhỏ nhất tìm bằng vét cạn `eval`. Dữ liệu: ví dụ đề, biên (`"1+1"`, số một chữ số, số dài nhất), và ngẫu nhiên seed cố định.

In [18]:
def is_valid_answer(expression, ans, best):
    v = evaluate(ans)
    return v is not None and ans.replace("(", "").replace(")", "") == expression and v == best

fns = {"eval": minimize_eval, "arith": minimize_arith, "precomputed": minimize_precomputed,
       "Solution": lambda e: Solution().minimizeResult(e), "SolutionArith": lambda e: SolutionArith().minimizeResult(e)}
edge = ["1+1", "9+9", "1+99999999", "99999999+1", "11+11", "5+5"]
for e in [x for x, _, _ in examples] + edge:
    best = evaluate(minimize_eval(e))
    for name, fn in fns.items():
        assert is_valid_answer(e, fn(e), best), (name, e, fn(e))
print("Ví dụ + biên: OK", {e: Solution().minimizeResult(e) for e in edge})

Ví dụ + biên: OK {'1+1': '(1+1)', '9+9': '(9+9)', '1+99999999': '(1+9999999)9', '99999999+1': '9(9999999+1)', '11+11': '1(1+1)1', '5+5': '(5+5)'}


In [19]:
for e in cases[:800]:
    best = min(evaluate(e[:i] + "(" + e[i:j + 1] + ")" + e[j + 1:]) for i in range(e.index("+")) for j in range(e.index("+") + 1, len(e)))
    for name, fn in fns.items():
        assert is_valid_answer(e, fn(e), best), (name, e)
print("800 biểu thức ngẫu nhiên: mọi cách cho đáp án hợp lệ và nhỏ nhất")

800 biểu thức ngẫu nhiên: mọi cách cho đáp án hợp lệ và nhỏ nhất


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Coi phần ngoài rỗng là 0 (hoặc `int('')` lỗi)

In [20]:
def wrong_empty_zero(expression):
    num1, num2 = expression.split("+"); best = None
    for i in range(len(num1)):
        for j in range(len(num2)):
            left = int(num1[:i] or 0); right = int(num2[j + 1:] or 0)         # SAI: rỗng phải là 1
            v = left * (int(num1[i:]) + int(num2[:j + 1])) * right
            if best is None or v < best[0]: best = (v, i, j)
    _, i, j = best
    return f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"

e = "247+38"
got = wrong_empty_zero(e)
print("sai:", got, "giá trị thật =", evaluate(got), "| đúng:", Solution().minimizeResult(e), "=", 170)

sai: (247+3)8 giá trị thật = 2000 | đúng: 2(47+38) = 170


In [21]:
print("sửa (rỗng -> 1):", minimize_arith(e))

sửa (rỗng -> 1): 2(47+38)


### 13.2 Lỗi 2 — Cho phép ngoặc phải ở bên trái dấu `+`

Bản sai duyệt mọi cặp `i < j` trên **toàn chuỗi**, sinh ra biểu thức không hợp lệ như `"2(4)7+38"` (hàm `evaluate` trả `None`).

In [22]:
def wrong_any_position(expression):
    best = None
    for i in range(len(expression)):
        for j in range(i, len(expression)):
            s = expression[:i] + "(" + expression[i:j + 1] + ")" + expression[j + 1:]
            try:
                v = eval(to_python(s))                                       # SAI: không kiểm vị trí của '+'
            except Exception:
                continue
            if best is None or v < best[0]: best = (v, s)
    return best[1]

got = wrong_any_position("247+38")
print("sai:", got, "| hợp lệ theo đề?", evaluate(got) is not None)

sai: 2(4)7+38 | hợp lệ theo đề? False


In [23]:
print("sửa:", Solution().minimizeResult("247+38"))

sửa: 2(47+38)


### Lỗi 3 — Dùng `eval` trực tiếp (không chèn `*`)

`eval("2(47+38)")` ném `TypeError` ("'int' object is not callable"), còn `eval("(247+3)8")` ném `SyntaxError`. Ta bắt lỗi để thấy thông báo (không để notebook dừng).

In [24]:
def wrong_eval(expression):
    p = expression.index("+")
    try:
        return min((eval(expression[:i] + "(" + expression[i:j + 1] + ")" + expression[j + 1:]),)
                   for i in range(p) for j in range(p + 1, len(expression)))
    except (TypeError, SyntaxError) as err:
        return f"{type(err).__name__}: {err}"

print("sai:", wrong_eval("247+38"))

sai: SyntaxError: invalid syntax (<string>, line 1)


In [25]:
print("sửa (chèn '*' hoặc tính số học):", minimize_eval("247+38"), "|", minimize_arith("247+38"))

sửa (chèn '*' hoặc tính số học): 2(47+38) | 2(47+38)


### Lỗi 4 — So sánh **chuỗi** thay vì **giá trị**

`min` trên chuỗi so theo thứ tự từ điển, chẳng liên quan tới giá trị biểu thức.

In [26]:
def wrong_string_min(expression):
    num1, num2 = expression.split("+")
    cands = [f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}" for i in range(len(num1)) for j in range(len(num2))]
    return min(cands)                                                        # SAI

got = wrong_string_min("12+34")
print("sai:", got, "=", evaluate(got), "| đúng:", Solution().minimizeResult("12+34"), "=", 20)

sai: (12+3)4 = 60 | đúng: 1(2+3)4 = 20


In [27]:
print("sửa (min theo giá trị):", min((evaluate(c), c) for c in ["(12+3)4", "(12+34)", "1(2+3)4", "1(2+34)"]))

sửa (min theo giá trị): (20, '1(2+3)4')


### Lỗi 5 — Bỏ sót trường hợp ngoặc bao trọn (`i = 0` hoặc `j` = cuối)

Duyệt `i` từ 1 hoặc `j` tới `len(num2) − 2` sẽ bỏ `"(999+999)"` — đáp án của ví dụ 3.

In [28]:
def wrong_skip_full(expression):
    num1, num2 = expression.split("+"); best = None
    for i in range(1, len(num1)):                                            # SAI: bỏ i = 0
        for j in range(len(num2) - 1):                                       # SAI: bỏ j cuối
            v = int(num1[:i]) * (int(num1[i:]) + int(num2[:j + 1])) * int(num2[j + 1:])
            if best is None or v < best[0]: best = (v, i, j)
    _, i, j = best
    return f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"

got = wrong_skip_full("999+999")
print("sai:", got, "=", evaluate(got), "| đúng:", "(999+999) = 1998")

sai: 9(99+99)9 = 16038 | đúng: (999+999) = 1998


In [29]:
print("sửa:", Solution().minimizeResult("999+999"))

sửa: (999+999)


### Lỗi 6 — Lệch chỉ số khi cắt `num2` (ngoặc trong rỗng)

Viết `num2[:j]` với `j` chạy từ 0 khiến phần trong bên phải **rỗng** ở `j = 0` → `int('')` ném `ValueError`; còn nếu "chữa cháy" bằng `or 0` thì sinh biểu thức không hợp lệ kiểu `"(247+)38"`. Tìm tự động biểu thức ngắn nhất mà bản sai trả về chuỗi không hợp lệ.

In [30]:
def wrong_off_by_one(expression):
    num1, num2 = expression.split("+"); best = None
    for i in range(len(num1)):
        for j in range(len(num2)):                                            # SAI: nên là num2[:j+1]
            left = int(num1[:i]) if i else 1
            right = int(num2[j:]) if j < len(num2) else 1
            v = left * (int(num1[i:]) + int(num2[:j] or 0)) * right
            if best is None or v < best[0]: best = (v, i, j)
    _, i, j = best
    return f"{num1[:i]}({num1[i:]}+{num2[:j]}){num2[j:]}"

bad = next(e for e in (a + "+" + b for la in range(1, 3) for lb in range(1, 3)
                       for a in map("".join, product("123", repeat=la)) for b in map("".join, product("123", repeat=lb)))
           if evaluate(wrong_off_by_one(e)) is None)
print("phản ví dụ:", bad, "-> bản sai trả", wrong_off_by_one(bad), "| hợp lệ?", evaluate(wrong_off_by_one(bad)) is not None)

phản ví dụ: 1+1 -> bản sai trả (1+)1 | hợp lệ? False


In [31]:
print("sửa:", Solution().minimizeResult(bad), "=", evaluate(Solution().minimizeResult(bad)))

sửa: (1+1) = 2


### Lỗi 7 — Trả về **giá trị** thay vì **chuỗi**

Đề yêu cầu trả về biểu thức (chuỗi), không phải số nhỏ nhất. Bản sai trả `170` cho `"247+38"` — LeetCode báo Wrong Answer (sai kiểu). Kiểm tra kiểu ngay trong bộ test (`isinstance(ans, str)`) để bắt lỗi này.

In [32]:
def wrong_return_value(expression):
    return best_value(expression)                                            # SAI: phải trả chuỗi

got = wrong_return_value("247+38")
print("sai:", got, type(got).__name__, "| đúng:", repr(Solution().minimizeResult("247+38")), "(str)")

sai: 170 int | đúng: '2(47+38)' (str)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Độ dài rất nhỏ** (≤ 10) ⇒ đừng ngại **liệt kê hết**; tập trung tính đúng từng trường hợp.
- **Phép nhân ngầm định** + phần rỗng ⇒ dùng **1** (trung hòa của nhân); phép cộng thì phần rỗng là 0.
- Tránh `eval` trong lời giải; nếu cần kiểm thử, tự viết bộ tính hoặc chèn toán tử rõ ràng.
- Bài "trả về bất kỳ đáp án tối ưu" ⇒ test bằng **giá trị** + **định dạng hợp lệ**.
- Đọc kỹ ràng buộc vị trí (ngoặc trái **trước** `+`, ngoặc phải **sau** `+`).

## 15. Biến thể của bài

1. **Tối đa hóa giá trị** (bài tập 5) — cùng liệt kê, đổi `min` thành `max`.
2. **Nhiều dấu `+`** với một cặp ngoặc: ngoặc có thể bao nhiều số hạng; vẫn liệt kê `(i, j)` nhưng phải tính tổng trong ngoặc và tổng ngoài ngoặc.
3. **LeetCode 241 — Different Ways to Add Parentheses:** mọi cách đặt ngoặc đầy đủ ⇒ chia để trị.
4. **LeetCode 282 — Expression Add Operators (Hard):** chèn toán tử giữa các chữ số.

Minh họa biến thể 3 (LeetCode 241, tự dựng, kiểm bằng liệt kê cây biểu thức nhỏ):

In [33]:
from functools import lru_cache
import re

def diff_ways(expression):
    tokens = re.findall(r"\d+|[+\-*]", expression)
    @lru_cache(maxsize=None)
    def solve(lo, hi):                       # các token số ở vị trí chẵn
        if lo == hi: return (int(tokens[lo]),)
        res = []
        for k in range(lo + 1, hi, 2):
            for a in solve(lo, k - 1):
                for b in solve(k + 1, hi):
                    res.append(a + b if tokens[k] == "+" else a - b if tokens[k] == "-" else a * b)
        return tuple(res)
    return sorted(solve(0, len(tokens) - 1))

print("2-1-1 ->", diff_ways("2-1-1"), "| 2*3-4*5 ->", diff_ways("2*3-4*5"))
assert diff_ways("2-1-1") == [0, 2] and diff_ways("2*3-4*5") == [-34, -14, -10, -10, 10]

2-1-1 -> [0, 2] | 2*3-4*5 -> [-34, -14, -10, -10, 10]


Minh họa biến thể 2 (tự dựng — **không phải đề LeetCode**, đối chiếu lại nếu dùng): biểu thức `a1+a2+…+ak`, đặt **một** cặp ngoặc sao cho ngoặc trái nằm trong một số hạng `as`, ngoặc phải nằm trong số hạng `at` với `s < t`; các số hạng nằm **giữa** `s` và `t` nằm trong ngoặc, các số hạng ngoài cộng bình thường. Chỉ phần chữ số ngay sát ngoặc mới nhân. Ta kiểm bằng `eval` (chèn `*`).

In [34]:
def minimize_multi(expression):
    terms = expression.split("+"); k = len(terms); best = None
    for s_ in range(k):
        for t in range(s_ + 1, k):
            for i in range(len(terms[s_])):
                for j in range(len(terms[t])):
                    L = int(terms[s_][:i]) if i else 1
                    R = int(terms[t][j + 1:]) if j + 1 < len(terms[t]) else 1
                    inner = int(terms[s_][i:]) + sum(int(x) for x in terms[s_ + 1:t]) + int(terms[t][:j + 1])
                    outside = sum(int(x) for x in terms[:s_]) + sum(int(x) for x in terms[t + 1:])
                    v = outside + L * inner * R
                    if best is None or v < best[0]:
                        best = (v, "+".join(terms[:s_] + [terms[s_][:i] + "(" + terms[s_][i:]] + terms[s_ + 1:t]
                                             + [terms[t][:j + 1] + ")" + terms[t][j + 1:]] + terms[t + 1:]))
    return best

def multi_eval(expr):
    out = []
    for idx, ch in enumerate(expr):
        if ch == "(" and idx > 0 and expr[idx - 1].isdigit(): out.append("*")
        out.append(ch)
        if ch == ")" and idx + 1 < len(expr) and expr[idx + 1].isdigit(): out.append("*")
    return eval("".join(out))

rngm = random.Random(7)
for _ in range(200):
    k = rngm.randint(2, 4)
    e = "+".join("".join(rngm.choice("123456789") for _ in range(rngm.randint(1, 3))) for _ in range(k))
    v, sexpr = minimize_multi(e)
    assert multi_eval(sexpr) == v and sexpr.replace("(", "").replace(")", "") == e
print("12+34+5 ->", minimize_multi("12+34+5"), "| 9+1+9 ->", minimize_multi("9+1+9"))

12+34+5 -> (25, '1(2+3)4+5') | 9+1+9 -> (19, '(9+1)+9')


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `eval` | dựng chuỗi Python, tính | `O(a·b)` × `eval` | `O(L)` | kiểm tra nhanh |
| 2. Số học | `L·(A+B)·R` | `O(a·b·L)` | `O(L)` | **nộp bài** |
| 3. Tiền tính | `int` mỗi lát cắt một lần | `O(L² + a·b)` | `O(L)` | khi `L` lớn hơn |
| 4. Bộ tính tự viết | tự phân tích `L(A+B)R` | `O(a·b·L)` | `O(L)` | kiểm định dạng |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `minimizeResult("1+1")` và giá trị.
2. **Tính tay:** liệt kê mọi cách đặt ngoặc của `"23+45"` và giá trị.
3. **Điền TODO:** `all_placements(expression)` trả về danh sách mọi biểu thức hợp lệ (có ngoặc).
4. **Tự viết lại:** viết lại Cách 2 không nhìn code.
5. **Mini challenge:** `maximizeResult(expression)` — đặt ngoặc để giá trị **lớn nhất**. Kiểm bằng vét cạn.

In [35]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def all_placements(expression):
    # TODO
    pass

In [36]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_minimize_result(expression):
    # TODO
    pass

In [37]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def maximize_result(expression):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Mỗi số chỉ một chữ số ⇒ chỉ một cách. (2) `"(1+1)"`. (3) Giá trị 2.

**Bài 2.** (1) `i ∈ {0, 1}`, `j ∈ {0, 1}`. (2) `(23+4)5 = 135`, `(23+45) = 68`, `2(3+4)5 = 70`, `2(3+45) = 96`. (3) Nhỏ nhất `68`.

**Bài 3.** (1) `p = expression.index('+')`. (2) `i` từ 0 tới `p − 1`, `j` từ `p + 1` tới `len − 1`. (3) Chèn `(` trước `i`, `)` sau `j`.

**Bài 4.** (1) `split('+')`. (2) Hai vòng `for`. (3) Phần rỗng = 1.

**Bài 5.** (1) Như Bài 4 nhưng `>`. (2) Thường ngoặc bao trọn cho giá trị nhỏ, ngoặc "giữa" cho giá trị lớn. (3) Kiểm bằng `evaluate` trên `all_placements`.

## 19. Đáp án tham khảo

In [38]:
# Bài 1, 2
assert Solution().minimizeResult("1+1") == "(1+1)" and evaluate("(1+1)") == 2
print("Bài 1: (1+1) = 2")
trace_arith("23+45")

Bài 1: (1+1) = 2
i=0 j=0: (23+4)5    = 1 × (23+4) × 5 =   135  <- kỷ lục mới
i=0 j=1: (23+45)     = 1 × (23+45) × 1 =    68  <- kỷ lục mới
i=1 j=0: 2(3+4)5    = 2 × (3+4) × 5 =    70
i=1 j=1: 2(3+45)     = 2 × (3+45) × 1 =    96


'(23+45)'

In [39]:
# Bài 3, 4
def all_placements_ans(expression):
    p = expression.index("+")
    return [expression[:i] + "(" + expression[i:j + 1] + ")" + expression[j + 1:] for i in range(p) for j in range(p + 1, len(expression))]

my_minimize_result_ans = minimize_arith
assert sorted(all_placements_ans("12+34")) == sorted(["(12+3)4", "(12+34)", "1(2+3)4", "1(2+34)"])
for e in cases[:300]:
    pl = all_placements_ans(e)
    assert all(evaluate(x) is not None for x in pl) and len(pl) == e.index("+") * (len(e) - e.index("+") - 1)
    assert evaluate(my_minimize_result_ans(e)) == min(evaluate(x) for x in pl)
print("Bài 3, 4: OK;", all_placements_ans("23+45"))

Bài 3, 4: OK; ['(23+4)5', '(23+45)', '2(3+4)5', '2(3+45)']


In [40]:
# Bài 5
def maximize_result_ans(expression):
    num1, num2 = expression.split("+"); best_v, best = -1, ""
    for i in range(len(num1)):
        for j in range(len(num2)):
            l = int(num1[:i]) if i else 1; r = int(num2[j + 1:]) if j + 1 < len(num2) else 1
            v = l * (int(num1[i:]) + int(num2[:j + 1])) * r
            if v > best_v: best_v, best = v, f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"
    return best

for e in cases[:300]:
    assert evaluate(maximize_result_ans(e)) == max(evaluate(x) for x in all_placements_ans(e))
print("Bài 5: OK; '247+38' ->", maximize_result_ans("247+38"), "=", evaluate(maximize_result_ans("247+38")))

Bài 5: OK; '247+38' -> (247+3)8 = 2000


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
num1, num2 = expression.split('+')
for i in range(len(num1)):                 # ngoặc trái trước num1[i]
    for j in range(len(num2)):             # ngoặc phải sau num2[j]
        L = int(num1[:i]) if i else 1
        R = int(num2[j+1:]) if j + 1 < len(num2) else 1
        val = L * (int(num1[i:]) + int(num2[:j+1])) * R
        # giữ (val, i, j) nhỏ nhất
```

In [41]:
def template(expression):
    num1, num2 = expression.split("+")
    v, i, j = min(((int(num1[:i]) if i else 1) * (int(num1[i:]) + int(num2[:j + 1])) * (int(num2[j + 1:]) if j + 1 < len(num2) else 1), i, j)
                  for i in range(len(num1)) for j in range(len(num2)))
    return f"{num1[:i]}({num1[i:]}+{num2[:j + 1]}){num2[j + 1:]}"
print([template(e) for e, _, _ in examples])

['2(47+38)', '1(2+3)4', '(999+999)']


**Checklist:**

- [ ] Ngoặc trái trong `num1` (kể cả trước chữ số đầu), ngoặc phải trong `num2` (kể cả sau chữ số cuối).
- [ ] Phần ngoài rỗng là **1**, không phải 0.
- [ ] Tính bằng số học; không `eval` chuỗi có nhân ngầm định.
- [ ] So sánh theo **giá trị**, không theo chuỗi.
- [ ] Không bỏ sót trường hợp ngoặc bao trọn `(num1+num2)`.
- [ ] Test bằng giá trị + định dạng hợp lệ (nhiều đáp án).